# perfora quickstart

**perfora turns a scan of a player-piano roll into data.** Every perforation
becomes a *note* — which lane it is in, where it starts and ends, measured in
millimetres — and every printed or handwritten label becomes a *text region*
with its position and meaning.

This notebook walks through one roll from start to finish. You do not need to
know Python: run each grey cell in order by clicking it and pressing
**Shift+Enter**. The only cell you may want to *change* is the next one.

A sample roll is included, so you can run the whole notebook before you have a
scan of your own.

---

## 1. Choose a roll

Three settings matter:

| Setting | What it means |
|---|---|
| `ROLL` | The file to read: a scan (`.png .jpg .tif`) or a video (`.mp4 .mov`). |
| `DPI` | The resolution your scanner was set to. perfora needs it to report **real millimetres**. |
| `LANES` | How many lanes (keyboard keys) the roll has, if you know. `0` means "measure it from the holes". |

To use your own scan: put the file in this folder (drag it into the file list on
the left), then replace `"sample_roll.png"` with its name, and set `DPI` to your
scanner's setting.

In [ ]:
ROLL = "sample_roll.png"   # the file to decode
DPI = 300                  # your scanner's resolution
LANES = 0                  # known lane count, or 0 to measure it

# ---------------------------------------------------------------------------
from pathlib import Path

import matplotlib.pyplot as plt

import perfora
from perfora.config import Config

roll_path = Path(ROLL)
if not roll_path.exists():
    raise FileNotFoundError(
        f"{roll_path} not found. Put the file in this folder "
        f"({Path.cwd()}) and set ROLL to its name."
    )
print(f"perfora {perfora.__version__} — reading {roll_path.name} at {DPI} dpi")

## 2. Decode it

perfora runs five steps in order:

1. **preprocess** — find the roll in the scan, straighten it, separate the
   perforations from the paper.
2. **holes** — find every individual perforation.
3. **lanes** — *measure* the lane spacing from those holes. perfora never
   assumes a roll standard; it derives the grid from your roll.
4. **notes** — join the perforations in each lane into notes.
5. **text** — find printed and handwritten labels (reading them needs an extra
   installed; without it, the locations are still recorded).

A `Session` runs those steps and keeps the intermediate results, so we can look
at each one afterwards.

In [ ]:
session = perfora.Session(
    perfora.ImageSource(roll_path, dpi=DPI),
    config=Config(n_lanes=LANES),
)
doc = session.run_all()

lanes = doc.lane_model
print(f"lane pitch      : {lanes.pitch_mm:.3f} mm")
print(f"lanes found     : {lanes.n_lanes}   (method: {lanes.method}, "
      f"confidence {lanes.confidence:.2f})")
print(f"notes assembled : {len(doc.notes)}")
print(f"text regions    : {len(doc.texts)}")
print(f"needs review    : {len(doc.review_queue)}")

## 3. Check the work, stage by stage

This is the important habit: **look at the pictures before trusting the
numbers.** The `lanes` picture is the one that matters most — if its vertical
lines sit on the columns of holes, everything downstream will be right.

In [ ]:
def show(stage, width=11):
    """Draw one stage's preview, with the numbers it reported underneath."""
    preview = session.preview(stage)
    if preview is None:
        print(f"{stage}: no preview available")
        return
    bgr = preview.rasterize(session.ctx.image)
    h, w = bgr.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(bgr[:, :, ::-1])  # OpenCV is BGR; matplotlib expects RGB
    plt.axis("off")
    summary = "  ".join(f"{k}={v}" for k, v in preview.summary.items())
    plt.title(f"{stage} — {summary}", fontsize=9)
    plt.show()


show("lanes")   # the measured lane grid: lines should sit on hole columns

In [ ]:
for stage in ("preprocess", "holes", "notes", "text"):
    show(stage)

## 4. The notes as a table

`doc.to_dataframe()` gives a spreadsheet-style table you can sort, filter, or
export to CSV / Excel.

- `lane` — which lane (0 is the first lane across the roll).
- `u_start_mm`, `u_end_mm` — where the note begins and ends **along** the roll.
- `v_center_mm` — where it sits **across** the roll.
- `length_mm` — how long the perforation is.
- `confidence` — how sure perfora is, from 0 to 1.

In [ ]:
df = doc.to_dataframe()
print(f"{len(df)} notes")
df.head(15)

In [ ]:
# Save the table as a spreadsheet file next to this notebook.
df.to_csv("notes.csv", index=False)
print(f"wrote notes.csv ({len(df)} rows)")

## 5. See it as a piano roll

Each note drawn as a horizontal bar: lane up the side, distance along the roll
across the bottom. This is the view that makes an error obvious — a lane that
should be silent, or a note that runs much longer than its neighbours.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for note in doc.notes:
    ax.barh(
        note.lane,
        note.u_end_mm - note.u_start_mm,
        left=note.u_start_mm,
        height=0.7,
        color=plt.cm.viridis(note.confidence),
    )
ax.set_xlabel("position along the roll, u (mm)")
ax.set_ylabel("lane")
ax.set_title(f"{roll_path.name} — {len(doc.notes)} notes (colour = confidence)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. What perfora was unsure about

perfora **never silently drops** a doubtful result. Anything below a confidence
threshold goes into the *review queue* — a list of things for a human to look
at, with the reason attached. An empty queue means nothing looked doubtful.

In [ ]:
if not doc.review_queue:
    print("nothing flagged — perfora was confident about every result")
for item in doc.review_queue:
    print(f"[{item.reason}] {item.ref_kind} #{item.ref_id} "
          f"(confidence {item.confidence:.2f}): {item.message}")

## 7. Save the result

The `.perfora.json` file is plain text you can open in any editor, and it is
**lossless**: reading it back gives exactly the same data. That makes it safe
for archiving.

In [ ]:
out_path = roll_path.with_suffix("").name + ".perfora.json"
perfora.write(doc, out_path)
print(f"wrote {out_path}")

# Proof that it round-trips: read it back and compare.
reloaded = perfora.read(out_path)
print("read back identical:", reloaded == doc)

## 8. If something looks wrong

Two knobs fix the great majority of problems. Change one, re-run **only** the
affected steps, and look again.

**The lane grid is off** (lines drift away from the hole columns): tell perfora
the true lane count. It is then treated as ground truth.

```python
LANES = 88   # in cell 1, then re-run from cell 1
```

**Notes that should be separate are merged** (or a dotted chain of holes came
out fragmented): change how big a gap along the roll is bridged into one note.

In [ ]:
import dataclasses

# Never merge two perforations into one note (0.0), or merge across bigger
# gaps for chain-perforated rolls (e.g. 1.5). Default is 0.5 mm.
# Config is immutable, so `replace` makes a copy with one field changed.
session.ctx.config = dataclasses.replace(session.ctx.config, bridge_gap_mm=0.0)
session.rerun_from("notes")   # redoes notes + text only, not the whole decode
doc = session.ctx.to_document()

print(f"notes now: {len(doc.notes)}")
show("notes")

---

## Where to go next

- **Every tunable setting**, with defaults and explanations:
  [Configuration reference](https://perfora.readthedocs.io/en/latest/configuration.html)
- **Unfamiliar words** (lane, pitch, `u`/`v`, calibration, review queue):
  [Glossary](https://perfora.readthedocs.io/en/latest/glossary.html)
- **What is in the output file**, field by field:
  [Output format](https://perfora.readthedocs.io/en/latest/output.html)
- **Reading the text on a roll** needs an OCR extra — see the
  [README](https://github.com/perfora-project/Perfora#reading-text-ocr).

Turning millimetres into seconds is deliberately left to you, because it depends
on how fast the roll was meant to play:

```python
seconds = doc.notes[0].duration_seconds(feed_rate_mm_per_s=180.0)
```

Found a bug, or a roll perfora handles badly? Please
[open an issue](https://github.com/perfora-project/Perfora/issues) — a
description of the roll and the numbers above are enough to start.